# DermAI — Training Pipeline (v2 — Phase-Split Edition)

### What changed from the previous notebook
1. **Steps 12 & 13 fixed** — `load_model` now uses `compile=False` + manual
   recompile, which is the only reliable way to load a `.h5` with a custom
   Focal Loss in Keras 3. The old `custom_objects={"focal_loss": ...}` key
   caused a `ValueError` because Keras 3's legacy H5 loader looks up the loss
   name in its own registry *before* checking `custom_objects`.
2. **Training split into two phases** — each fits inside a free T4 GPU hour:
   - **Step 11-A  →  Phase 1** (head only, backbone frozen, 10 epochs)
   - **Step 11-B  →  Phase 2** (backbone fine-tune, run in a *new* session)
   Both steps auto-resume from the Drive checkpoint, so nothing is lost
   between sessions.
3. **Steps 7–10 are skipped on repeat runs** — data already on Drive/Colab
   disk is detected and reused automatically.

### Recommended workflow
| Session | Steps to run |
|---------|-------------|
| First   | 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10 → **11-A** |
| Second  | 1 → 2 → 3 → 4 → 5 → **11-B** → 12 → 13 → 14 |

---
### v3 → v4 fixes (this notebook)
1. **Resumable Phase 2** — tracks epoch progress in `phase2_progress.json`; re-running
   STEP 11-B after an interrupt resumes at the correct epoch instead of restarting the
   15-epoch budget and LR warmup from zero.
2. **Warmup-resume LR bug fixed** — resuming past the warmup window now compiles
   directly at the peak fine-tune LR instead of silently training at the warmup-start LR forever.
3. **Native `.keras` format** everywhere instead of legacy `.h5` (with a one-time
   migration cell, STEP 11-B0, for existing `.h5` checkpoints).
4. **Unfreeze budget fixed** — `N_UNFREEZE` now comes from `hp["fine_tune_layers"]`
   (single source of truth) and only layers with actual trainable weights count
   against it, so parameter-free layers like `GlobalAveragePooling2D` no longer
   silently eat one of your 20 "unfrozen" slots.
5. **Removed dead code** (unused `backbone_layers` list in the old unfreeze block).
6. **Graceful `KeyboardInterrupt` handling** in both Phase 1 and Phase 2 — a
   disconnect no longer ends in an ugly traceback, and progress is always saved.
7. **No more stale `phase2_done.json`** — it's only written after a run that
   actually completes, so an interrupted run can't leave behind a flag that
   makes a later report look "final" when it isn't.
8. **Checkpoint mtime printed** at every load/save point (Phase 1, Phase 2, report,
   verify) so you can always confirm which model you're actually looking at.


In [1]:
# ════════════════════════════════════════════════════════════
# STEP 1 — Verify GPU
# ════════════════════════════════════════════════════════════
import tensorflow as tf
print("TF Version:", tf.__version__)
print("GPU Found :", tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), "❌ No GPU! Go to Runtime → Change runtime type → T4 GPU"
print("✅ GPU confirmed")

TF Version: 2.20.0
GPU Found : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✅ GPU confirmed


In [2]:
# ════════════════════════════════════════════════════════════
# STEP 2 — Mount Google Drive
# ════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/DermAI"
os.makedirs(DRIVE_ROOT, exist_ok=True)
print("✅ Drive mounted. Save folder:", DRIVE_ROOT)

Mounted at /content/drive
✅ Drive mounted. Save folder: /content/drive/MyDrive/DermAI


In [3]:
# ════════════════════════════════════════════════════════════
# STEP 3 — Install packages
# ════════════════════════════════════════════════════════════
!pip install -q tensorflow opencv-python-headless scikit-learn \
    matplotlib pandas numpy Pillow reportlab scipy kagglehub
print("✅ Packages installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 33.5 MB/s eta 0:00:00
✅ Packages installed


In [4]:
# ════════════════════════════════════════════════════════════
# STEP 4 — Upload project zip (auto-loads from Drive on repeat runs)
# ════════════════════════════════════════════════════════════
import os, zipfile, shutil
from google.colab import files

DRIVE_ROOT  = "/content/drive/MyDrive/DermAI"
PROJECT_DIR = "/content/Zip_project"
DRIVE_ZIP   = f"{DRIVE_ROOT}/Zip_project.zip"

os.makedirs(PROJECT_DIR, exist_ok=True)

if os.path.isfile(DRIVE_ZIP):
    print("✅ Found Zip_project.zip in Drive — extracting...")
    with zipfile.ZipFile(DRIVE_ZIP, "r") as z:
        z.extractall("/content/")
    print("✅ Extracted from Drive")
else:
    print("Zip_project.zip not found in Drive.")
    print("Upload it now:")
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    with zipfile.ZipFile(fname, "r") as z:
        z.extractall("/content/")
    shutil.copy(fname, DRIVE_ZIP)
    print("✅ Uploaded, extracted, and saved to Drive for next time")

print("Project files:", os.listdir(PROJECT_DIR))

✅ Found Zip_project.zip in Drive — extracting...
✅ Extracted from Drive
Project files: ['app.py', 'dermai.db', '.streamlit', 'requirements.txt', 'database', 'dataset_report', 'training_report', 'assets', 'DermAI_Training_FIXED.ipynb', 'pages', 'skin_dataset', 'dataset_explorer.py', '.gitignore', 'prepare_dataset.py', 'train_model.py', 'utils', 'models']


In [5]:
# ════════════════════════════════════════════════════════════
# STEP 5 — Patch train_model.py: fix WarmUpLR learning-rate bug
# ════════════════════════════════════════════════════════════
# On recent TF/Keras, optimizer.learning_rate is a tf.Variable that must
# be updated with .assign() — not tf.keras.backend.set_value().
# This patch makes WarmUpLR robust across all Keras versions.

content = open("/content/Zip_project/train_model.py").read()

OLD_LR = '''    def on_epoch_begin(self, epoch, logs=None):
        if epoch < self.warmup_epochs:
            lr = self.start_lr + (self.target_lr - self.start_lr) * (epoch / self.warmup_epochs)
            tf.keras.backend.set_value(self.model.optimizer.learning_rate, lr)
            print(f"\n  [WarmUp] epoch {epoch+1}: lr={lr:.2e}")'''

NEW_LR = '''    def on_epoch_begin(self, epoch, logs=None):
        if epoch < self.warmup_epochs:
            lr = self.start_lr + (self.target_lr - self.start_lr) * (epoch / self.warmup_epochs)
            opt = self.model.optimizer
            if hasattr(opt, "learning_rate"):
                if hasattr(opt.learning_rate, "assign"):
                    opt.learning_rate.assign(lr)
                else:
                    tf.keras.backend.set_value(opt.learning_rate, lr)
            elif hasattr(opt, "lr"):
                tf.keras.backend.set_value(opt.lr, lr)
            print(f"\n  [WarmUp] epoch {epoch+1}: lr={lr:.2e}")'''

if OLD_LR in content:
    content = content.replace(OLD_LR, NEW_LR)
    print("✅ WarmUpLR patched for Keras 3 compatibility")
elif "opt.learning_rate.assign(lr)" in content:
    print("✅ WarmUpLR already patched — nothing to do")
else:
    print("⚠️  WarmUpLR pattern not found — check train_model.py manually")

with open("/content/Zip_project/train_model.py", "w") as f:
    f.write(content)

✅ WarmUpLR already patched — nothing to do


In [ ]:
# ════════════════════════════════════════════════════════════
# STEP 5B — Patch train_model.py: fix oversampling + mislabeled training time
# ════════════════════════════════════════════════════════════
# Three fixes applied here (post-evaluation of skin_model v2 results):
#
# 1) oversample_targets: "df" was boosted 167 -> 600 (3.6x duplication),
#    which caused the model to over-predict "df" at inference time
#    (test precision 0.17 despite recall 0.53). Reduced to a more moderate
#    ~2x duplication for df/vasc, and added mild oversampling for bcc/akiec
#    (previously untouched despite also being minority, high-risk classes).
#
# 2) save_metrics_txt(): the old report labeled the *evaluation* loop's
#    elapsed time as "Training time", producing a nonsensical "1.6 min"
#    figure for an EfficientNetB3 model. Now correctly split into
#    "Training time" (real Phase 1 + Phase 2 wall-clock, from the time-log
#    JSONs saved in Steps 11-A/11-B) and "Evaluation time" (test-set
#    inference only).

content = open("/content/Zip_project/train_model.py").read()

# ── Fix 1: oversampling targets ───────────────────────────────────────────
OLD_OVERSAMPLE = '''    "oversample_targets": {
        6: 600,   # df   → was ~167 train samples → boost to 600
        5: 400,   # vasc → was ~177 train samples → boost to 400
    },'''

NEW_OVERSAMPLE = '''    "oversample_targets": {
        3: 500,   # bcc   -> mild boost (malignant class, was unboosted)
        4: 400,   # akiec -> mild boost (pre-malignant class, was unboosted)
        5: 300,   # vasc  -> was 400 (2.3x); reduced to ~1.7x duplication
        6: 350,   # df    -> was 600 (3.6x); reduced to ~2.1x duplication
    },'''

if OLD_OVERSAMPLE in content:
    content = content.replace(OLD_OVERSAMPLE, NEW_OVERSAMPLE)
    print("✅ oversample_targets patched (df/vasc reduced, bcc/akiec added)")
elif '"3: 500"' in content or "3: 500," in content:
    print("✅ oversample_targets already patched — nothing to do")
else:
    print("⚠️  oversample_targets pattern not found — check train_model.py manually")

# ── Fix 2: mislabeled training time in save_metrics_txt() ────────────────
OLD_SIG = "def save_metrics_txt(y_true, y_pred, y_probs, num_classes, roc_aucs, elapsed):"
NEW_SIG = '''def save_metrics_txt(y_true, y_pred, y_probs, num_classes, roc_aucs, elapsed,
                      train_elapsed=None):'''

OLD_LINES = '''    kw   = dict(average="weighted", zero_division=0)
    kw_m = dict(average="macro",    zero_division=0)
    lines = [
        "TRAINING RESULTS REPORT (UPGRADED MODEL)",
        "=" * 64,
        f"Training time   : {elapsed/60:.1f} min",
        f"Num classes     : {num_classes}",'''

NEW_LINES = '''    kw   = dict(average="weighted", zero_division=0)
    kw_m = dict(average="macro",    zero_division=0)
    train_time_str = f"{train_elapsed/60:.1f} min" if train_elapsed else "not recorded"
    lines = [
        "TRAINING RESULTS REPORT (UPGRADED MODEL)",
        "=" * 64,
        f"Training time   : {train_time_str}  (Phase 1 + Phase 2, wall-clock)",
        f"Evaluation time : {elapsed/60:.1f} min  (test-set inference only)",
        f"Num classes     : {num_classes}",'''

if OLD_SIG in content:
    content = content.replace(OLD_SIG, NEW_SIG)
    content = content.replace(OLD_LINES, NEW_LINES)
    print("✅ save_metrics_txt() patched — training vs evaluation time now separate")
elif "train_elapsed=None" in content:
    print("✅ save_metrics_txt() already patched — nothing to do")
else:
    print("⚠️  save_metrics_txt() pattern not found — check train_model.py manually")

with open("/content/Zip_project/train_model.py", "w") as f:
    f.write(content)


In [6]:
# ════════════════════════════════════════════════════════════
# STEP 6 — Configure Kaggle API (needed for HAM10000 + ISIC downloads)
# ════════════════════════════════════════════════════════════
from google.colab import files
import os

if not os.path.isfile(os.path.expanduser("~/.kaggle/kaggle.json")):
    print("Upload your kaggle.json API key:")
    uploaded = files.upload()
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    os.rename("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    print("✅ Kaggle configured")
else:
    print("✅ Kaggle already configured")

Upload your kaggle.json API key:


Saving kaggle.json to kaggle.json
✅ Kaggle configured


In [7]:
# ════════════════════════════════════════════════════════════
# STEP 7 — Download HAM10000 into skin_dataset/
# ════════════════════════════════════════════════════════════
import os, shutil, kagglehub

DATASET_DIR = "/content/Zip_project/skin_dataset"
os.makedirs(DATASET_DIR, exist_ok=True)

# Skip if already downloaded
ham_part1 = os.path.join(DATASET_DIR, "HAM10000_images_part_1")
ham_part2 = os.path.join(DATASET_DIR, "HAM10000_images_part_2")
if os.path.isdir(ham_part1) and len(os.listdir(ham_part1)) > 5000:
    print(f"✅ HAM10000 already present ({len(os.listdir(ham_part1))} + {len(os.listdir(ham_part2))} images)")
else:
    print("Downloading HAM10000 dataset...")
    download_path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
    print(f"Dataset downloaded to: {download_path}")
    print(f"Copying files into {DATASET_DIR} ...")
    for item in os.listdir(download_path):
        s = os.path.join(download_path, item)
        d = os.path.join(DATASET_DIR, item)
        if os.path.isdir(s):
            if os.path.exists(d):
                shutil.rmtree(d)
            shutil.copytree(s, d)
        else:
            shutil.copy2(s, d)
    print("✅ HAM10000 ready in:", DATASET_DIR)

print("Contents:", os.listdir(DATASET_DIR))

Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.
Dataset downloaded to: /kaggle/input/skin-cancer-mnist-ham10000
Copying files into /content/Zip_project/skin_dataset ...
✅ HAM10000 ready in: /content/Zip_project/skin_dataset
Contents: ['ham10000_images_part_2', 'merged_dataset.csv', 'hmnist_28_28_RGB.csv', 'HAM10000_metadata.csv', 'hmnist_8_8_L.csv', 'HAM10000_images_part_2', 'HAM10000_images_part_1', 'ham10000_images_part_1', 'hmnist_28_28_L.csv', 'hmnist_8_8_RGB.csv']


In [8]:
# ════════════════════════════════════════════════════════════
# STEP 8 — Download ISIC 2019 into skin_dataset/ISIC_2019_Training_Input/
# ════════════════════════════════════════════════════════════
import os, shutil, kagglehub

DATASET_DIR = "/content/Zip_project/skin_dataset"
ISIC_DEST   = os.path.join(DATASET_DIR, "ISIC_2019_Training_Input")

# Skip if already downloaded
if os.path.isdir(ISIC_DEST) and len(os.listdir(ISIC_DEST)) >= 25000:
    print(f"✅ ISIC 2019 already present ({len(os.listdir(ISIC_DEST))} images)")
else:
    print("Accessing ISIC 2019 via kagglehub...")
    isic_path = kagglehub.dataset_download("andrewmvd/isic-2019")
    print("Cache path:", isic_path)
    os.makedirs(ISIC_DEST, exist_ok=True)

    zip_file           = os.path.join(isic_path, "ISIC_2019_Training_Input.zip")
    cache_img_dir      = os.path.join(isic_path, "ISIC_2019_Training_Input", "ISIC_2019_Training_Input")
    cache_img_dir_flat = os.path.join(isic_path, "ISIC_2019_Training_Input")

    if os.path.exists(zip_file):
        print("Unzipping images (this may take a few minutes)...")
        !unzip -q "{zip_file}" -d "{ISIC_DEST}_tmp"
        nested  = os.path.join(f"{ISIC_DEST}_tmp", "ISIC_2019_Training_Input")
        src_dir = nested if os.path.isdir(nested) else f"{ISIC_DEST}_tmp"
        for f in os.listdir(src_dir):
            shutil.move(os.path.join(src_dir, f), os.path.join(ISIC_DEST, f))
        shutil.rmtree(f"{ISIC_DEST}_tmp", ignore_errors=True)
    elif os.path.isdir(cache_img_dir):
        print("Copying images from kagglehub cache (nested dir)...")
        for f in os.listdir(cache_img_dir):
            shutil.copy2(os.path.join(cache_img_dir, f), os.path.join(ISIC_DEST, f))
    elif os.path.isdir(cache_img_dir_flat):
        print("Copying images from kagglehub cache (flat dir)...")
        for f in os.listdir(cache_img_dir_flat):
            src = os.path.join(cache_img_dir_flat, f)
            if os.path.isfile(src) and f.lower().endswith((".jpg", ".jpeg", ".png")):
                shutil.copy2(src, os.path.join(ISIC_DEST, f))
    else:
        print("❌ Could not locate ISIC images in kagglehub cache.")

    final_count = len(os.listdir(ISIC_DEST)) if os.path.isdir(ISIC_DEST) else 0
    print(f"✅ {final_count} ISIC images in {ISIC_DEST}")

    # Copy ground-truth / metadata CSVs
    for csv_file in ["ISIC_2019_Training_GroundTruth.csv", "ISIC_2019_Training_Metadata.csv"]:
        src = os.path.join(isic_path, csv_file)
        dst = os.path.join(DATASET_DIR, csv_file)
        if os.path.exists(src):
            shutil.copy2(src, dst)
            print(f"Copied {csv_file}")
        elif os.path.exists(dst):
            print(f"{csv_file} already present")
        else:
            print(f"⚠️  {csv_file} not found in kagglehub cache")

Accessing ISIC 2019 via kagglehub...
Using Colab cache for faster access to the 'isic-2019' dataset.
Cache path: /kaggle/input/isic-2019
Copying images from kagglehub cache (nested dir)...
✅ 25333 ISIC images in /content/Zip_project/skin_dataset/ISIC_2019_Training_Input
Copied ISIC_2019_Training_GroundTruth.csv
Copied ISIC_2019_Training_Metadata.csv


In [9]:
# ════════════════════════════════════════════════════════════
# STEP 9 — Regenerate merged_dataset.csv with CORRECT paths
# ════════════════════════════════════════════════════════════
# Re-runs prepare_dataset.py to build a fresh CSV with paths
# correct for this Colab environment (ignores Windows paths in zip).
%cd /content/Zip_project
!python prepare_dataset.py

/content/Zip_project
════════════════════════════════════════════════════════════
  DermAI Monitor — Dataset Preparation Script
  HAM10000 + Selective ISIC 2019 merge
════════════════════════════════════════════════════════════

[STEP 1] Loading HAM10000 ...
  Loaded 10015 HAM10000 images
    nv        6705
    mel       1113
    bkl       1099
    bcc        514
    akiec      327
    vasc       142
    df         115

[STEP 2] Loading ISIC 2019 ...
  Total ISIC 2019 images: 25331
  Removed 10015 duplicates (HAM images already in ISIC 2019)
  Genuinely new ISIC images: 15316
  Skipped 9716 images (NV, BCC, AK, UNK — not needed)
  New images to add: 5600
    MEL    → mel       3409 new images
    BKL    → bkl       1525 new images
    DF     → df         124 new images
    VASC   → vasc       111 new images
    SCC    → akiec      431 new images

[STEP 3] Merging datasets ...

════════════════════════════════════════════════════════════
  MERGED DATASET SUMMARY
════════════════════════

In [10]:
# ════════════════════════════════════════════════════════════
# STEP 10 — Verify dataset is ready for training
# ════════════════════════════════════════════════════════════
import pandas as pd, os

merged = pd.read_csv("/content/Zip_project/skin_dataset/merged_dataset.csv")
print(merged["dx"].value_counts().to_string())
print(f"Total rows: {len(merged)}")

resolved = merged["image_path"].apply(os.path.isfile).sum()
print(f"\nImages resolved on disk: {resolved}/{len(merged)}")
assert resolved > 0, "❌ No images found on disk — check Steps 7–9"
print("✅ Dataset ready for training")

dx
nv       6705
mel      4522
bkl      2624
akiec     758
bcc       514
vasc      253
df        239
Total rows: 15615

Images resolved on disk: 15615/15615
✅ Dataset ready for training


In [ ]:
# ════════════════════════════════════════════════════════════
# STEP 10B — Save exact final dataset composition (for paper + slides)
# ════════════════════════════════════════════════════════════
# Freezes the EXACT per-class counts used for this training run, plus the
# resulting train/val/test split sizes, into a text file in the report
# folder. Use these numbers in your research paper / slides instead of
# re-deriving them later from a possibly-stale CSV.

import os, json
from sklearn.model_selection import train_test_split
import train_model as tm

DRIVE_ROOT   = "/content/drive/MyDrive/DermAI"
DRIVE_REPORT = f"{DRIVE_ROOT}/training_report"
os.makedirs(DRIVE_REPORT, exist_ok=True)

all_paths, all_labels = tm.load_data()

train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_paths, all_labels, test_size=0.30, random_state=42, stratify=all_labels
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.50, random_state=42, stratify=temp_labels
)

from collections import Counter
def counts_by_class(labels):
    c = Counter(labels)
    return {tm.CLASS_NAMES[k]: c.get(k, 0) for k in range(tm.NUM_CLASSES)}

summary_lines = [
    "FINAL DATASET COMPOSITION (frozen at training time)",
    "=" * 64,
    f"Total images loaded : {len(all_paths)}",
    f"Train split          : {len(train_paths)}",
    f"Val split             : {len(val_paths)}",
    f"Test split            : {len(test_paths)}",
    "",
    "Per-class counts — TOTAL:",
]
for k, v in counts_by_class(all_labels).items():
    summary_lines.append(f"  {k:<8}: {v}")
summary_lines += ["", "Per-class counts — TRAIN (pre-oversampling):"]
for k, v in counts_by_class(train_labels).items():
    summary_lines.append(f"  {k:<8}: {v}")
summary_lines += ["", "Per-class counts — VAL:"]
for k, v in counts_by_class(val_labels).items():
    summary_lines.append(f"  {k:<8}: {v}")
summary_lines += ["", "Per-class counts — TEST:"]
for k, v in counts_by_class(test_labels).items():
    summary_lines.append(f"  {k:<8}: {v}")

out_path = f"{DRIVE_REPORT}/dataset_final_summary.txt"
with open(out_path, "w") as f:
    f.write("\n".join(summary_lines))
print(f"✅ Saved {out_path}")
print("\n".join(summary_lines))


## Training — Split into Two Phases

Training EfficientNetB3 fully takes ~90 min on a free T4.  
Since each Colab session is ~60 min, training is split:

| Step | Phase | What trains | Epochs | Session |
|------|-------|-------------|--------|---------|
| 11-A | Phase 1 | Head only (backbone frozen) | 10 | First |
| 11-B | Phase 2 | Last 20 backbone layers | 10 | Second |

Both steps **resume from the Drive checkpoint** automatically.  
Run **11-A** first session. After timeout, start a new session and run **11-B**.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 11-A — PHASE 1: Train head only (backbone frozen)
# ══════════════════════════════════════════════════════════════════════════════
# • Runs 10 epochs with backbone frozen (only the Dense head trains).
# • If a checkpoint already exists in Drive, resumes from it.
# • Best weights are saved to Drive every epoch — session timeout safe.
# • FIX: now saves in native .keras format instead of legacy .h5.
# • After this session ends, run STEP 11-B0 (once) then STEP 11-B.
# ══════════════════════════════════════════════════════════════════════════════
import sys, os, importlib, json
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

PROJECT_DIR  = "/content/Zip_project"
DRIVE_ROOT   = "/content/drive/MyDrive/DermAI"
DRIVE_MODEL  = f"{DRIVE_ROOT}/skin_model.keras"
DRIVE_REPORT = f"{DRIVE_ROOT}/training_report"
os.makedirs(DRIVE_ROOT,   exist_ok=True)
os.makedirs(DRIVE_REPORT, exist_ok=True)

%cd /content/Zip_project
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import train_model as tm
importlib.reload(tm)

hp = tm.HYPERPARAMS

# ── Build Focal Loss instance ─────────────────────────────────────────────────
focal_loss = tm.SparseCategoricalFocalLoss(
    gamma=hp["focal_gamma"],
    alpha=hp["focal_alpha"],
    name="focal_loss",
)

# ── Load or build model ───────────────────────────────────────────────────────
if os.path.isfile(DRIVE_MODEL):
    print(f"✅ Checkpoint found — resuming from {DRIVE_MODEL}")
    model = tf.keras.models.load_model(
        DRIVE_MODEL,
        custom_objects={"SparseCategoricalFocalLoss": tm.SparseCategoricalFocalLoss},
        compile=False,   # skip Keras registry lookup — avoids ValueError
    )
    # Recompile with same Phase 1 settings
    model.compile(
        optimizer=tf.keras.optimizers.Adam(hp["lr_phase1"]),
        loss=focal_loss,
        metrics=["accuracy"],
    )
    print("  Recompiled with Adam lr=", hp["lr_phase1"])
else:
    print("No checkpoint found — building model from scratch")
    model, _ = tm.build_model(hp, tm.NUM_CLASSES, focal_loss=focal_loss)

model.summary(show_trainable=True)

# ── Rebuild dataset (same random_state=42 → same split every time) ───────────
print("\nLoading dataset...")
all_paths, all_labels = tm.load_data()
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_paths, all_labels, test_size=0.30, random_state=42, stratify=all_labels
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.50, random_state=42, stratify=temp_labels
)
print(f"Train: {len(train_paths)}  Val: {len(val_paths)}  Test: {len(test_paths)}")

# Class weights
unique_labels     = np.unique(train_labels)
cw                = compute_class_weight("balanced", classes=unique_labels, y=train_labels)
class_weight_dict = dict(zip(unique_labels.tolist(), cw.tolist()))

# Oversampling
train_paths, train_labels = tm.oversample_rare_classes(
    train_paths, train_labels, hp["oversample_targets"]
)
print(f"Training set after oversampling: {len(train_paths)} samples")

# Sequences
bs         = hp["batch_size"]
train_seq  = tm.SkinSequence(train_paths, train_labels, bs, hp, augment=True, aug_intensity=1.5)
val_seq    = tm.SkinSequence(val_paths,   val_labels,   bs, hp, augment=False)
metrics_cb = tm.MetricsCallback(val_seq, every_n_epochs=1)

# Callbacks
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
cb_phase1 = [
    tm.WarmUpLR(hp["warmup_epochs"], hp["lr_warmup_start"], hp["lr_phase1"]),
    EarlyStopping(patience=hp["patience"], restore_best_weights=True, verbose=1),
    ModelCheckpoint(DRIVE_MODEL, save_best_only=True, verbose=1),
    ReduceLROnPlateau(factor=hp["lr_reduce_factor"],
                      patience=hp["lr_reduce_patience"], min_lr=1e-8, verbose=1),
    metrics_cb,
]

# ── PHASE 1 TRAINING ─────────────────────────────────────────────────────────
print("\n[PHASE 1] Head training — EfficientNetB3 backbone frozen ...")
print(f"Trainable params: {sum(tf.size(v).numpy() for v in model.trainable_variables):,}")

import time as _time
_phase1_start = _time.time()

try:
    history1 = model.fit(
        train_seq,
        validation_data=val_seq,
        epochs=10,                   # fits in ~55 min on T4
        callbacks=cb_phase1,
        class_weight=class_weight_dict,
        verbose=1,
    )
    _phase1_completed = True
except KeyboardInterrupt:
    print("\n⚠️  Phase 1 interrupted — best weights up to this point are already "
          "saved to Drive (ModelCheckpoint saves every improved epoch).")
    print("   Just re-run this cell to resume from the saved checkpoint.")
    _phase1_completed = False

_phase1_elapsed = _time.time() - _phase1_start
print(f"\n[PHASE 1] Wall-clock time this run: {_phase1_elapsed/60:.1f} min")

if _phase1_completed:
    # Save phase1 status flag to Drive so Phase 2 knows Phase 1 is done.
    # Accumulates elapsed time across restarts instead of overwriting it.
    _prev_seconds = 0
    _flag_path = f"{DRIVE_ROOT}/phase1_done.json"
    if os.path.isfile(_flag_path):
        with open(_flag_path) as f:
            _prev_seconds = json.load(f).get("phase1_train_seconds", 0)

    with open(_flag_path, "w") as f:
        json.dump({"phase1_epochs_completed": 10,
                   "best_val_f1": max(metrics_cb.metric_history["val_f1"]),
                   "phase1_train_seconds": _prev_seconds + _phase1_elapsed}, f, indent=2)

    import datetime
    print(f"\n✅ Phase 1 complete — checkpoint at {DRIVE_MODEL}")
    print(f"   Checkpoint last modified: {datetime.datetime.fromtimestamp(os.path.getmtime(DRIVE_MODEL))}")
    print("   Start a NEW Colab session and run STEP 11-B0 then STEP 11-B for Phase 2 fine-tuning.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 11-B0 — One-time migration: legacy .h5 checkpoint → native .keras format
# ══════════════════════════════════════════════════════════════════════════════
# Run this once. If you already have skin_model.h5 on Drive but no
# skin_model.keras yet, this loads the old .h5 and re-saves it in the native
# Keras format (removes the HDF5-legacy warning spam and is more robust for
# models with custom layers/losses going forward). Safe to re-run — it is a
# no-op if skin_model.keras already exists.
# ══════════════════════════════════════════════════════════════════════════════
import os, sys, importlib
import tensorflow as tf

PROJECT_DIR  = "/content/Zip_project"
DRIVE_ROOT   = "/content/drive/MyDrive/DermAI"
DRIVE_MODEL_OLD = f"{DRIVE_ROOT}/skin_model.h5"
DRIVE_MODEL_NEW = f"{DRIVE_ROOT}/skin_model.keras"

if os.path.isfile(DRIVE_MODEL_NEW):
    print(f"✅ {DRIVE_MODEL_NEW} already exists — nothing to migrate")
elif os.path.isfile(DRIVE_MODEL_OLD):
    %cd /content/Zip_project
    if PROJECT_DIR not in sys.path:
        sys.path.insert(0, PROJECT_DIR)
    import train_model as tm
    importlib.reload(tm)

    print(f"Loading legacy checkpoint: {DRIVE_MODEL_OLD}")
    _legacy_model = tf.keras.models.load_model(
        DRIVE_MODEL_OLD,
        custom_objects={"SparseCategoricalFocalLoss": tm.SparseCategoricalFocalLoss},
        compile=False,
    )
    _legacy_model.save(DRIVE_MODEL_NEW)
    print(f"✅ Migrated → {DRIVE_MODEL_NEW}")
    print("   (Old .h5 left untouched on Drive as a backup — safe to delete later.)")
else:
    print("No checkpoint found yet (neither .h5 nor .keras) — nothing to migrate. "
          "Run STEP 11-A (Phase 1) first.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 11-B — PHASE 2: Fine-tune backbone (run in a NEW Colab session)
# ══════════════════════════════════════════════════════════════════════════════
# Prerequisites: Steps 1–5B and STEP 11-B0 (migration) must be run first.
# This cell loads the checkpoint from Drive, unfreezes the last N backbone
# layers, and fine-tunes at a small learning rate.
#
# FIXES in this version:
#   1) .keras format instead of legacy .h5.
#   2) RESUMABLE: tracks epochs completed in phase2_progress.json. If this
#      cell is interrupted (Colab disconnect, KeyboardInterrupt), re-running
#      it picks up at the correct epoch instead of restarting the LR warmup
#      and the full 15-epoch budget from zero every time.
#   3) On resume, skips the warmup ramp (already done in a prior session) and
#      compiles directly at the peak fine-tune LR instead of silently
#      training at the warmup-start LR forever — this was a latent bug.
#   4) Unfreeze logic no longer wastes a slot on parameter-free layers
#      (e.g. GlobalAveragePooling2D) — only layers that actually own
#      trainable weights count against the unfreeze budget, so
#      hp["fine_tune_layers"] means what it says.
#   5) Removed dead code (unused backbone_layers list).
#   6) model.fit() is wrapped in try/except so a KeyboardInterrupt exits
#      cleanly instead of an ugly traceback, and still records progress.
#   7) phase2_done.json / config.json are only ever written after a run
#      that actually completes or exhausts its epoch budget — no more
#      stale leftover flags from a previous, unrelated completed run.
# ══════════════════════════════════════════════════════════════════════════════
import sys, os, importlib, json, datetime
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

PROJECT_DIR   = "/content/Zip_project"
DRIVE_ROOT    = "/content/drive/MyDrive/DermAI"
DRIVE_MODEL   = f"{DRIVE_ROOT}/skin_model.keras"
DRIVE_REPORT  = f"{DRIVE_ROOT}/training_report"
PROGRESS_PATH = f"{DRIVE_ROOT}/phase2_progress.json"
TOTAL_PHASE2_EPOCHS = 15
os.makedirs(DRIVE_ROOT,   exist_ok=True)
os.makedirs(DRIVE_REPORT, exist_ok=True)

# Safety check
assert os.path.isfile(DRIVE_MODEL), "❌ No checkpoint found — run STEP 11-A and STEP 11-B0 first"
phase1_flag = f"{DRIVE_ROOT}/phase1_done.json"
if os.path.isfile(phase1_flag):
    with open(phase1_flag) as f:
        info = json.load(f)
    print(f"✅ Phase 1 confirmed done — best val F1 was {info.get('best_val_f1', '?'):.4f}")
else:
    print("⚠️  phase1_done.json not found — make sure Phase 1 actually completed")

print(f"Checkpoint last modified: {datetime.datetime.fromtimestamp(os.path.getmtime(DRIVE_MODEL))}")

# ── Resume state ───────────────────────────────────────────────────────────
if os.path.isfile(PROGRESS_PATH):
    with open(PROGRESS_PATH) as f:
        _progress = json.load(f)
    initial_epoch = _progress.get("epochs_completed", 0)
    if initial_epoch >= TOTAL_PHASE2_EPOCHS:
        print(f"✅ phase2_progress.json shows {initial_epoch}/{TOTAL_PHASE2_EPOCHS} epochs already "
              "completed — nothing left to train. Skip to STEP 12 to generate the report, or "
              "delete phase2_progress.json to force a restart.")
    else:
        print(f"↻ Resuming Phase 2 from epoch {initial_epoch}/{TOTAL_PHASE2_EPOCHS} "
              f"(interrupted or multi-session run detected)")
else:
    initial_epoch = 0
    print(f"▶ Starting Phase 2 fresh (0/{TOTAL_PHASE2_EPOCHS} epochs completed)")

%cd /content/Zip_project
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import train_model as tm
importlib.reload(tm)

hp = tm.HYPERPARAMS

# ── Focal Loss ────────────────────────────────────────────────────────────────
focal_loss = tm.SparseCategoricalFocalLoss(
    gamma=hp["focal_gamma"],
    alpha=hp["focal_alpha"],
    name="focal_loss",
)

# ── Load checkpoint ────────────────────────────────────────────────────────────
print(f"\nLoading checkpoint from {DRIVE_MODEL} ...")
model = tf.keras.models.load_model(
    DRIVE_MODEL,
    custom_objects={"SparseCategoricalFocalLoss": tm.SparseCategoricalFocalLoss},
    compile=False,   # avoids the focal_loss ValueError in Keras 3
)
print("✅ Checkpoint loaded")
print("   Input shape :", model.input_shape)
print("   Output shape:", model.output_shape)

# ── Unfreeze last N backbone layers (FIX: single source of truth, and only
#    layers that actually own trainable weights count against the budget) ───
N_UNFREEZE = hp["fine_tune_layers"]   # was a separate hardcoded 20 before — now
                                       # matches whatever train_model.py declares
HEAD_LAYER_NAMES = ("dense", "dense_1", "batch_normalization", "dropout")

for layer in model.layers:
    layer.trainable = False

unfrozen = 0
for layer in reversed(model.layers):
    if layer.name in HEAD_LAYER_NAMES:
        layer.trainable = True   # always keep head trainable, never counted
        continue
    if unfrozen < N_UNFREEZE:
        layer.trainable = True
        if layer.weights:   # only layers with actual trainable params count
            unfrozen += 1   # (skips GlobalAveragePooling2D, Activation, Add, etc.)

total_trainable = sum(tf.size(v).numpy() for v in model.trainable_variables)
print(f"✅ Unfroze {unfrozen} weight-bearing backbone layers + custom head")
print(f"   Trainable params now: {total_trainable:,}")
print("   Unfrozen layers:", [l.name for l in model.layers if l.trainable])

# ── Recompile ──────────────────────────────────────────────────────────────────
LR_PHASE2_START = 1e-6
LR_PHASE2_PEAK  = hp["lr_phase2"]   # 1e-5

if initial_epoch == 0:
    # Fresh start — begin at the warmup-start LR, WarmUpLR ramps it up.
    model.compile(
        optimizer=tf.keras.optimizers.Adam(LR_PHASE2_START),
        loss=focal_loss,
        metrics=["accuracy"],
    )
    print(f"   Recompiled with Adam lr={LR_PHASE2_START} (warmup start)")
    print(f"   Will ramp to {LR_PHASE2_PEAK} over first 2 epochs")
else:
    # FIX: resuming past the warmup window — compile directly at the peak LR.
    # Previously, resuming would silently keep training at LR_PHASE2_START
    # forever because WarmUpLR only fires while epoch < warmup_epochs, and a
    # freshly reloaded optimizer has no memory of ever having ramped up.
    model.compile(
        optimizer=tf.keras.optimizers.Adam(LR_PHASE2_PEAK),
        loss=focal_loss,
        metrics=["accuracy"],
    )
    print(f"   Resuming past warmup — compiled directly at peak lr={LR_PHASE2_PEAK}")

# ── Rebuild dataset ───────────────────────────────────────────────────────────
print("\nLoading dataset...")
all_paths, all_labels = tm.load_data()
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_paths, all_labels, test_size=0.30, random_state=42, stratify=all_labels
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.50, random_state=42, stratify=temp_labels
)
print(f"Train: {len(train_paths)}  Val: {len(val_paths)}  Test: {len(test_paths)}")

unique_labels     = np.unique(train_labels)
cw                = compute_class_weight("balanced", classes=unique_labels, y=train_labels)
class_weight_dict = dict(zip(unique_labels.tolist(), cw.tolist()))

train_paths, train_labels = tm.oversample_rare_classes(
    train_paths, train_labels, hp["oversample_targets"]
)

bs         = hp["batch_size"]
train_seq  = tm.SkinSequence(train_paths, train_labels, bs, hp, augment=True, aug_intensity=0.6)
val_seq    = tm.SkinSequence(val_paths,   val_labels,   bs, hp, augment=False)
metrics_cb = tm.MetricsCallback(val_seq, every_n_epochs=1)

# ── Progress-tracking callback (FIX: makes resume possible) ──────────────────
class ProgressSaver(tf.keras.callbacks.Callback):
    def __init__(self, path, total_epochs):
        super().__init__()
        self.path = path
        self.total_epochs = total_epochs

    def on_epoch_end(self, epoch, logs=None):
        with open(self.path, "w") as f:
            json.dump({"epochs_completed": epoch + 1,
                       "total_epochs": self.total_epochs}, f, indent=2)

# Callbacks
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

warmup_phase2 = tm.WarmUpLR(
    warmup_epochs=2,
    start_lr=LR_PHASE2_START,
    target_lr=LR_PHASE2_PEAK,
)

cb_phase2 = [
    warmup_phase2,
    EarlyStopping(patience=hp["patience"], restore_best_weights=True, verbose=1),
    ModelCheckpoint(DRIVE_MODEL, save_best_only=True, verbose=1),
    ReduceLROnPlateau(factor=hp["lr_reduce_factor"],
                      patience=hp["lr_reduce_patience"], min_lr=1e-9, verbose=1),
    metrics_cb,
    ProgressSaver(PROGRESS_PATH, TOTAL_PHASE2_EPOCHS),
]

# ── PHASE 2 TRAINING ─────────────────────────────────────────────────────────
print(f"\n[PHASE 2] Fine-tuning last {N_UNFREEZE} EfficientNetB3 layers "
      f"(epochs {initial_epoch} → {TOTAL_PHASE2_EPOCHS}) ...")

import time as _time
_phase2_start = _time.time()
_phase2_completed = False

try:
    history2 = model.fit(
        train_seq,
        validation_data=val_seq,
        epochs=TOTAL_PHASE2_EPOCHS,
        initial_epoch=initial_epoch,   # FIX: resumes at the right epoch
        callbacks=cb_phase2,
        class_weight=class_weight_dict,
        verbose=1,
    )
    _phase2_completed = True
except KeyboardInterrupt:
    print("\n⚠️  Phase 2 interrupted.")
    print("   Best weights up to the last improved epoch are already saved to Drive.")
    print(f"   Progress saved to {PROGRESS_PATH} — just re-run this cell to resume "
          "at the right epoch and LR, no manual bookkeeping needed.")

_phase2_elapsed = _time.time() - _phase2_start
print(f"\n[PHASE 2] Wall-clock time this run: {_phase2_elapsed/60:.1f} min")
print(f"Checkpoint last modified: {datetime.datetime.fromtimestamp(os.path.getmtime(DRIVE_MODEL))}")

if _phase2_completed:
    # FIX: only write phase2_done.json / config.json when this run actually
    # completed (or exhausted its epoch budget) — no more stale flags left
    # over from an earlier run silently masking an interrupted later one.
    T_opt = tm.calibrate_temperature(model, val_seq, tm.NUM_CLASSES)
    config = {
        "preprocess":  "efficientnet",
        "img_size":    list(tm.IMG_SIZE),
        "num_classes": tm.NUM_CLASSES,
        "class_names": tm.CLASS_NAMES[:tm.NUM_CLASSES],
        "trained_at":  datetime.datetime.now().isoformat(),
        "temperature": T_opt,
        "backbone":    "EfficientNetB3",
        "loss":        f"FocalLoss(gamma={hp['focal_gamma']})",
        "phase2_done": True,
    }
    config_path = DRIVE_MODEL.replace(".keras", "_config.json")
    with open(config_path, "w") as f:
        json.dump(config, f, indent=2)

    _phase1_seconds = 0
    if os.path.isfile(phase1_flag):
        with open(phase1_flag) as f:
            _phase1_seconds = json.load(f).get("phase1_train_seconds", 0)

    # Accumulate phase-2 time across resumed sessions instead of overwriting.
    _prev_phase2_seconds = 0
    _phase2_flag = f"{DRIVE_ROOT}/phase2_done.json"
    if os.path.isfile(_phase2_flag):
        with open(_phase2_flag) as f:
            _prev_phase2_seconds = json.load(f).get("phase2_train_seconds", 0)
    _total_phase2_seconds = _prev_phase2_seconds + _phase2_elapsed
    _total_train_seconds  = _phase1_seconds + _total_phase2_seconds

    with open(_phase2_flag, "w") as f:
        json.dump({
            "phase2_done": True,
            "phase2_train_seconds": _total_phase2_seconds,
            "total_train_seconds": _total_train_seconds,
        }, f, indent=2)
    print(f"   Total training time (Phase 1 + Phase 2): {_total_train_seconds/60:.1f} min")

    print(f"\n✅ Phase 2 complete — best model saved to {DRIVE_MODEL}")
    print(f"   Temperature T = {T_opt:.4f} saved to config.json")
    print("   Now run STEP 12 to generate the full evaluation report.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 12 — GENERATE REPORT from checkpoint
# ══════════════════════════════════════════════════════════════════════════════
# Run after Phase 1 OR Phase 2 — works on whichever checkpoint is in Drive.
# FIX: .keras path + prints checkpoint mtime and Phase-2 progress so you can
#      confirm you're evaluating the model you think you are, before trusting
#      the numbers in the report.
# ══════════════════════════════════════════════════════════════════════════════
import sys, os, importlib, time, json, datetime
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split

PROJECT_DIR   = "/content/Zip_project"
DRIVE_ROOT    = "/content/drive/MyDrive/DermAI"
DRIVE_MODEL   = f"{DRIVE_ROOT}/skin_model.keras"
DRIVE_REPORT  = f"{DRIVE_ROOT}/training_report"
PROGRESS_PATH = f"{DRIVE_ROOT}/phase2_progress.json"
os.makedirs(DRIVE_REPORT, exist_ok=True)

assert os.path.isfile(DRIVE_MODEL), f"❌ No checkpoint at {DRIVE_MODEL} — run Step 11-A first"
print(f"✅ Found checkpoint: {DRIVE_MODEL}")
print(f"   Last modified: {datetime.datetime.fromtimestamp(os.path.getmtime(DRIVE_MODEL))}")

# FIX: surface whether Phase 2 actually finished, and how far it got, so a
# report generated after an interrupted run doesn't get mistaken for final.
if os.path.isfile(PROGRESS_PATH):
    with open(PROGRESS_PATH) as f:
        _progress = json.load(f)
    _ep = _progress.get("epochs_completed", 0)
    _tot = _progress.get("total_epochs", "?")
    print(f"   Phase 2 progress: {_ep}/{_tot} epochs completed"
          + ("" if str(_ep) == str(_tot) else "  ⚠️  NOT the full budget — this may not be final"))
else:
    print("   ⚠️  No phase2_progress.json found — this checkpoint may only reflect Phase 1")

# ── Load model — compile=False avoids Keras 3 ValueError on focal_loss lookup ─
%cd /content/Zip_project
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
import train_model as tm
importlib.reload(tm)

focal_loss = tm.SparseCategoricalFocalLoss(
    gamma=tm.HYPERPARAMS["focal_gamma"],
    alpha=tm.HYPERPARAMS["focal_alpha"],
    name="focal_loss",
)

model = tf.keras.models.load_model(
    DRIVE_MODEL,
    custom_objects={"SparseCategoricalFocalLoss": tm.SparseCategoricalFocalLoss},
    compile=False,
)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=focal_loss,
    metrics=["accuracy"],
)
print("✅ Model loaded successfully")
print("   Input shape :", model.input_shape)
print("   Output shape:", model.output_shape)

tm.REPORT_DIR = DRIVE_REPORT

# ── Rebuild exact same test split (random_state=42) ──────────────────────────
print("\nLoading dataset...")
all_paths, all_labels = tm.load_data()
_, temp_paths, _, temp_labels = train_test_split(
    all_paths, all_labels, test_size=0.30, random_state=42, stratify=all_labels
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.50, random_state=42, stratify=temp_labels
)
print(f"Test: {len(test_paths)} | Val: {len(val_paths)}")

hp       = tm.HYPERPARAMS
test_seq = tm.SkinSequence(test_paths, test_labels, hp["batch_size"], hp, augment=False)
val_seq  = tm.SkinSequence(val_paths,  val_labels,  hp["batch_size"], hp, augment=False)

# ── Evaluate ──────────────────────────────────────────────────────────────────
print("\nEvaluating on test set (takes a few minutes)...")
t0 = time.time()
y_true, y_pred, y_probs, roc_aucs = tm.evaluate_model(model, test_seq, tm.NUM_CLASSES)

tm.plot_confusion_matrix(y_true, y_pred, tm.NUM_CLASSES)
tm.plot_roc_curves(y_true, y_probs, tm.NUM_CLASSES, roc_aucs)
tm.plot_per_class_metrics(y_true, y_pred, tm.NUM_CLASSES)

_train_elapsed = None
_phase2_flag = f"{DRIVE_ROOT}/phase2_done.json"
_phase1_flag = f"{DRIVE_ROOT}/phase1_done.json"
if os.path.isfile(_phase2_flag):
    with open(_phase2_flag) as f:
        _train_elapsed = json.load(f).get("total_train_seconds")
elif os.path.isfile(_phase1_flag):
    with open(_phase1_flag) as f:
        _train_elapsed = json.load(f).get("phase1_train_seconds")

tm.save_metrics_txt(y_true, y_pred, y_probs, tm.NUM_CLASSES, roc_aucs,
                     elapsed=time.time()-t0, train_elapsed=_train_elapsed)

# Temperature calibration
T_opt = tm.calibrate_temperature(model, val_seq, tm.NUM_CLASSES)

# Update config JSON
config_path = DRIVE_MODEL.replace(".keras", "_config.json")
if os.path.isfile(config_path):
    with open(config_path) as f:
        cfg = json.load(f)
    cfg["temperature"] = T_opt
    with open(config_path, "w") as f:
        json.dump(cfg, f, indent=2)
    print(f"Updated temperature={T_opt:.4f} in config.json")

print(f"\n✅ Report saved to {DRIVE_REPORT}/")
print("Files:", sorted(os.listdir(DRIVE_REPORT)))


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 13 — Verify model loads correctly (from Drive)
# ══════════════════════════════════════════════════════════════════════════════
# FIX: .keras path. Verifies output sums to ~1.0 (softmax sanity check).
# ══════════════════════════════════════════════════════════════════════════════
import sys, os, importlib, datetime
import tensorflow as tf
import numpy as np

PROJECT_DIR = "/content/Zip_project"
DRIVE_MODEL = "/content/drive/MyDrive/DermAI/skin_model.keras"

%cd /content/Zip_project
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
import train_model as tm
importlib.reload(tm)

focal_loss = tm.SparseCategoricalFocalLoss(
    gamma=tm.HYPERPARAMS["focal_gamma"],
    alpha=tm.HYPERPARAMS["focal_alpha"],
    name="focal_loss",
)

model = tf.keras.models.load_model(
    DRIVE_MODEL,
    custom_objects={"SparseCategoricalFocalLoss": tm.SparseCategoricalFocalLoss},
    compile=False,
)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=focal_loss,
    metrics=["accuracy"],
)

print("✅ Model loaded")
print(f"   Last modified: {datetime.datetime.fromtimestamp(os.path.getmtime(DRIVE_MODEL))}")
print("   Input shape :", model.input_shape)
print("   Output shape:", model.output_shape)

dummy = np.zeros((1, *model.input_shape[1:]), dtype=np.float32)
out   = model.predict(dummy, verbose=0)
total = round(float(out.sum()), 4)
print(f"   Output sum  : {total}  (should be ~1.0)")
assert 0.99 <= total <= 1.01, f"❌ Softmax sum {total} is wrong — model may be corrupt"
print("✅ Model is valid and ready for Streamlit app")


In [15]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 14 — Download model + config to your laptop
# ══════════════════════════════════════════════════════════════════════════════
from google.colab import files
import os

DRIVE_ROOT = "/content/drive/MyDrive/DermAI"
for fname in ["skin_model.h5", "skin_model_config.json"]:
    fpath = f"{DRIVE_ROOT}/{fname}"
    if os.path.isfile(fpath):
        files.download(fpath)
        print(f"✅ Downloading {fname}")
    else:
        print(f"⚠️  {fname} not found in Drive")
print("Check your browser Downloads folder")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloading skin_model.h5
⚠️  skin_model_config.json not found in Drive
Check your browser Downloads folder
